In [71]:
##### Compiles csv of training data for regions in dataset and does final cleaning

import os
import pandas as pd
from pathlib import Path
import numpy as np
import geopandas as gpd
from functools import reduce

In [72]:
### Set-up

# Get the current working directory
cd = Path.cwd().parent 

# folder of sub-national predictors 
predictors = Path(f'{cd}/Data/Clean/Predictors/Vectors')

# country averages
country_avg = pd.read_csv(f'{cd}/Data/Clean/Predictors/country_averages.csv')

# intensities 
capital = pd.read_csv(f"{cd}/Data/Clean/Intensities/subnational_capital_intensity.csv")
labor = pd.read_csv(f"{cd}/Data/Clean/Intensities/subnational_labor_intensity.csv")

# lat / lon 
lat_lon = pd.read_csv(f"{cd}/Data/Clean/Predictors/Vectors/lat_lon.csv")

# save path
save_path_capital_abs = f'{cd}/Data/Clean/Training_data/capital_absolute.csv'
save_path_labor_abs = f'{cd}/Data/Clean/Training_data/labor_absolute.csv'

save_path_capital_rtv = f'{cd}/Data/Clean/Training_data/capital_relative.csv'
save_path_labor_rtv = f'{cd}/Data/Clean/Training_data/labor_relative.csv'

### Assemble vectors for absolute training data

In [73]:
##### Merge all predictors into one df (dropping all duplicate PROJ_ID entries first)
dfs = []
for f in os.listdir(predictors):
    if f.endswith(".csv"):
        df = pd.read_csv(os.path.join(predictors, f))
        dupes = df['PROJ_ID'].duplicated().sum()
        if dupes > 0:
            df = df.drop_duplicates(subset='PROJ_ID')
        dfs.append(df)

predictors_df = reduce(lambda left, right: pd.merge(left, right, on='PROJ_ID', how='outer'), dfs)

In [74]:
### Merge with capital and labor 

### Capital
capital = capital.drop_duplicates(subset='PROJ_ID')
capital_merge = capital.merge(predictors_df, on='PROJ_ID', how='left')

### Labor
labor = labor.drop_duplicates(subset='PROJ_ID')
labor_merge = labor.merge(predictors_df, on='PROJ_ID', how='left')

In [75]:
### Save
capital_merge.to_csv(save_path_capital_abs, index=False)
labor_merge.to_csv(save_path_labor_abs, index=False)

### Assemble vectors for relative training data

In [76]:
##### Merge with country averages 

# Add country ID column
capital_merge['ISO3'] = capital_merge['PROJ_ID'].str[:3]
labor_merge['ISO3'] = labor_merge['PROJ_ID'].str[:3]

# Merge with country data
country_avg = country_avg.rename(columns=lambda c: c if c == 'ISO3' else f'country_{c}')

capital_merge_country = capital_merge.merge(country_avg, on='ISO3', how='left')
labor_merge_country = labor_merge.merge(country_avg, on='ISO3', how='left')


In [77]:
##### Re-scale variables for log1p transformation (see method memo)

## capital intensities: 
# USD per USD -> USD per million USD
capital_merge_country['capital_intensity_USD_per_million_USD'] = capital_merge_country['capital_intensity_USD_per_USD'] * 1e6
capital_merge_country['country_capital_intensity_USD_per_million_USD'] = capital_merge_country['country_capital_intensity_USD_per_USD'] * 1e6
labor_merge_country['country_capital_intensity_USD_per_million_USD'] = labor_merge_country['country_capital_intensity_USD_per_USD'] * 1e6
# USD per tonne -> USD per million tonne
capital_merge_country['capital_intensity_USD_per_million_tonne'] = capital_merge_country['capital_intensity_USD_per_tonne'] * 1e6
capital_merge_country['country_capital_intensity_USD_per_million_tonne'] = capital_merge_country['country_capital_intensity_USD_per_tonne'] * 1e6
labor_merge_country['country_capital_intensity_USD_per_million_tonne'] = labor_merge_country['country_capital_intensity_USD_per_tonne'] * 1e6

## labor intensities: 
# jobs per million USD -> stays the same
# jobs per tonne -> jobs per million tonnes
labor_merge_country['labor_intensity_jobs_per_million_tonne'] = labor_merge_country['labor_intensity_jobs_per_tonne'] * 1e6
labor_merge_country['country_labor_intensity_jobs_per_million_tonne'] = labor_merge_country['country_labor_intensity_jobs_per_tonne'] * 1e6
capital_merge_country['country_labor_intensity_jobs_per_million_tonne'] = capital_merge_country['country_labor_intensity_jobs_per_tonne'] * 1e6

## production intensities:
# USD production per HA -> USD production per million HA
capital_merge_country['USD_production_per_million_HA'] = capital_merge_country['USD_production_per_HA'] * 1e6
labor_merge_country['USD_production_per_million_HA'] = labor_merge_country['USD_production_per_HA'] * 1e6
capital_merge_country['country_USD_production_per_million_HA'] = capital_merge_country['country_USD_production_per_HA'] * 1e6
labor_merge_country['country_USD_production_per_million_HA'] = labor_merge_country['country_USD_production_per_HA'] * 1e6
# tonnes production per HA -> tonnes production per million HA
capital_merge_country['tonnes_production_per_million_HA'] = capital_merge_country['tonnes_production_per_HA'] * 1e6
labor_merge_country['tonnes_production_per_million_HA'] = labor_merge_country['tonnes_production_per_HA'] * 1e6
capital_merge_country['country_tonnes_production_per_million_HA'] = capital_merge_country['country_tonnes_production_per_HA'] * 1e6
labor_merge_country['country_tonnes_production_per_million_HA'] = labor_merge_country['country_tonnes_production_per_HA'] * 1e6

## density variables:
# people per km2 -> people per 100 km2
# animals per km2 -> animals per 100 km2
density_vars = ['pop_density_people_per_km2',
    'buffalo_density_per_km2',
    'chicken_density_per_km2',
    'cattle_density_per_km2',
    'goats_density_per_km2',
    'pigs_density_per_km2',
    'sheep_density_per_km2',
    'livestock_density_LU_per_km2'
]
for df in [capital_merge_country, labor_merge_country]:
    for var in density_vars:
        # region level
        new_var = var.replace('_per_km2', '_per_100_km2')
        df[new_var] = df[var] * 100
        # country level
        country_var = f'country_{var}'
        country_new_var = f'country_{new_var}'
        df[country_new_var] = df[country_var] * 100

## share variables
# 1% is currently 0.01 -> 1 (100 base)
share_vars = ['female_share',
    'cereals_share_production_USD',
    'fibres_share_production_USD',
    'fruits_share_production_USD',
    'oilcrops_share_production_USD',
    'pulses_share_production_USD',
    'roots_tubers_share_production_USD', 
    'rest_of_crops_share_production_USD', 
    'sugar_crops_share_production_USD', 
    'vegetables_share_production_USD', 
    'rubber_share_production_USD', 
    'ruminants_share_production_USD', 
    'monogastrics_share_production_USD', 
    'poultry_share_production_USD',  
    'share_large_field', 
    'share_medium_field', 
    'share_small_field',
    'share_vsmall_field', 
    'share_with_nightlights'
]

for df in [capital_merge_country, labor_merge_country]:
    for var in share_vars:
        # region level
        new_var = var.replace('share', 'share_base_100')
        df[new_var] = df[var] * 100
        # country level
        country_var = f'country_{var}'
        country_new_var = f'country_{new_var}'
        df[country_new_var] = df[country_var] * 100

pct_vars = ['pct_cropland_irrigated',
    'pct_GDP_ag']

for df in [capital_merge_country, labor_merge_country]:
    for var in pct_vars:
        # region level
        new_var = var.replace('pct', 'pct_base_100')
        df[new_var] = df[var] * 100
        # country level
        country_var = f'country_{var}'
        country_new_var = f'country_{new_var}'
        df[country_new_var] = df[country_var] * 100

## Child Dependency Ratio: 
# # of people aged (0-14) / # of people aged 15+ -> # of people aged (0-14) / 100 people aged 15+ 
capital_merge_country['child_dependency_ratio_per_hundred'] = capital_merge_country['child_dependency_ratio'] * 100
labor_merge_country['child_dependency_ratio_per_hundred'] = labor_merge_country['child_dependency_ratio'] * 100
capital_merge_country['country_child_dependency_ratio_per_hundred'] = capital_merge_country['country_child_dependency_ratio'] * 100
labor_merge_country['country_child_dependency_ratio_per_hundred'] = labor_merge_country['country_child_dependency_ratio'] * 100

## SOC, average travel time, growing season length, GDP pc, slope, crop intensity
# no transformation

#### Drop old columns
cols_to_drop = [
    'capital_intensity_USD_per_USD',
    'country_capital_intensity_USD_per_USD',
    'capital_intensity_USD_per_tonne',
    'country_capital_intensity_USD_per_tonne',
    'labor_intensity_jobs_per_tonne',
    'country_labor_intensity_jobs_per_tonne',
    'USD_production_per_HA',
    'country_USD_production_per_HA',
    'tonnes_production_per_HA',
    'country_tonnes_production_per_HA',
    'child_dependency_ratio',
    'country_child_dependency_ratio'
]

# Add density variables 
cols_to_drop.extend(density_vars)
cols_to_drop.extend([f'country_{v}' for v in density_vars])

# Add share variables 
cols_to_drop.extend(share_vars)
cols_to_drop.extend([f'country_{v}' for v in share_vars])

cols_to_drop.extend(pct_vars)
cols_to_drop.extend([f'country_{v}' for v in pct_vars])


# Drop columns
for df in [capital_merge_country, labor_merge_country]:
    df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

In [78]:
##### Do log1p transformations 

log_columns_cap = ['SOC',
       'average_travel_time_city', 'average_travel_time_port', 'growing_season_length_days',
       'GDP_pc', 'slope', 'crop_intensity',
       'capital_intensity_USD_per_million_USD',
       'capital_intensity_USD_per_million_tonne',
       'USD_production_per_million_HA',
       'tonnes_production_per_million_HA',
       'pop_density_people_per_100_km2',
       'buffalo_density_per_100_km2',
       'chicken_density_per_100_km2',
       'cattle_density_per_100_km2',
       'goats_density_per_100_km2',
       'pigs_density_per_100_km2',
       'sheep_density_per_100_km2', 'livestock_density_LU_per_100_km2',
       'female_share_base_100',
       'cereals_share_base_100_production_USD',
       'fibres_share_base_100_production_USD',
       'fruits_share_base_100_production_USD',
       'oilcrops_share_base_100_production_USD',
       'pulses_share_base_100_production_USD',
       'roots_tubers_share_base_100_production_USD',
       'rest_of_crops_share_base_100_production_USD',
       'sugar_crops_share_base_100_production_USD',
       'vegetables_share_base_100_production_USD',
       'rubber_share_base_100_production_USD',
       'ruminants_share_base_100_production_USD',
       'monogastrics_share_base_100_production_USD',
       'poultry_share_base_100_production_USD',
       'child_dependency_ratio_per_hundred', 'pct_base_100_GDP_ag',  
       'share_base_100_large_field', 
       'share_base_100_medium_field', 
       'share_base_100_small_field',
       'share_base_100_vsmall_field', 
       'share_base_100_with_nightlights', 
       'pct_base_100_cropland_irrigated']

log_columns_lab = ['SOC',
       'average_travel_time_city', 'average_travel_time_port', 'growing_season_length_days',
       'GDP_pc', 'slope', 'crop_intensity',
       'labor_intensity_jobs_per_million_tonne',
       'labor_intensity_jobs_per_million_USD',
       'USD_production_per_million_HA',
       'tonnes_production_per_million_HA',
       'pop_density_people_per_100_km2',
       'buffalo_density_per_100_km2',
       'chicken_density_per_100_km2',
       'cattle_density_per_100_km2',
       'goats_density_per_100_km2',
       'pigs_density_per_100_km2',
       'sheep_density_per_100_km2', 'livestock_density_LU_per_100_km2',
       'female_share_base_100',
       'cereals_share_base_100_production_USD',
       'fibres_share_base_100_production_USD',
       'fruits_share_base_100_production_USD',
       'oilcrops_share_base_100_production_USD',
       'pulses_share_base_100_production_USD',
       'roots_tubers_share_base_100_production_USD',
       'rest_of_crops_share_base_100_production_USD',
       'sugar_crops_share_base_100_production_USD',
       'vegetables_share_base_100_production_USD',
       'rubber_share_base_100_production_USD',
       'ruminants_share_base_100_production_USD',
       'monogastrics_share_base_100_production_USD',
       'poultry_share_base_100_production_USD',
       'child_dependency_ratio_per_hundred', 'pct_base_100_GDP_ag',  
       'share_base_100_large_field', 
       'share_base_100_medium_field', 
       'share_base_100_small_field',
       'share_base_100_vsmall_field', 
       'share_base_100_with_nightlights', 
       'pct_base_100_cropland_irrigated']

for col in log_columns_cap:
    capital_merge_country[f"rtv_log_{col}"] = np.log1p(capital_merge_country[col]) - np.log1p(capital_merge_country[f"country_{col}"])
capital_merge_country = capital_merge_country.drop(columns=[col for col in log_columns_cap] + [f"country_{col}" for col in log_columns_cap])

for col in log_columns_lab:
    labor_merge_country[f"rtv_log_{col}"] = np.log1p(labor_merge_country[col]) - np.log1p(labor_merge_country[f"country_{col}"])
labor_merge_country = labor_merge_country.drop(columns=[col for col in log_columns_lab] + [f"country_{col}" for col in log_columns_lab])


/var/folders/48/ky2jtbmj31bfj15cr5gq480w0000gn/T/ipykernel_31797/3046899563.py:76: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  capital_merge_country[f"rtv_log_{col}"] = np.log1p(capital_merge_country[col]) - np.log1p(capital_merge_country[f"country_{col}"])
/var/folders/48/ky2jtbmj31bfj15cr5gq480w0000gn/T/ipykernel_31797/3046899563.py:76: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  capital_merge_country[f"rtv_log_{col}"] = np.log1p(capital_merge_country[col]) - np.log1p(capital_merge_country[f"country_{col}"])
/var/folders/4

In [79]:
##### Probability variables 
capital_merge_country['rtv_probability_survival_land_use_objective'] = capital_merge_country['probability_survival_land_use_objective'] - capital_merge_country['country_probability_survival_land_use_objective']
capital_merge_country['rtv_probability_economic_land_use_objective'] = capital_merge_country['probability_economic_land_use_objective'] - capital_merge_country['country_probability_economic_land_use_objective']
capital_merge_country = capital_merge_country.drop(columns=['probability_economic_land_use_objective', 'probability_survival_land_use_objective', 'country_probability_survival_land_use_objective', 'country_probability_economic_land_use_objective'])

labor_merge_country['rtv_probability_survival_land_use_objective'] = labor_merge_country['probability_survival_land_use_objective'] - labor_merge_country['country_probability_survival_land_use_objective']
labor_merge_country['rtv_probability_economic_land_use_objective'] = labor_merge_country['probability_economic_land_use_objective'] - labor_merge_country['country_probability_economic_land_use_objective']
labor_merge_country = labor_merge_country.drop(columns=['probability_economic_land_use_objective', 'probability_survival_land_use_objective', 'country_probability_survival_land_use_objective', 'country_probability_economic_land_use_objective'])


In [80]:
##### Drop unneeeded cols

col_to_drop_cap = ['GEO_ID_NAME', 'ag_capital_stock_USD_nominal',
       'total_production_USD', 'total_production_t',
       'other_share_production_USD', 'share_vlarge_field', 'lon', 'lat',
       'country_labor_intensity_jobs_per_million_USD', 'ISO3',
       'country_other_share_production_USD', 'country_share_vlarge_field',
       'country_labor_intensity_jobs_per_million_tonne' ]
col_to_drop_lab = ['GEO_ID_NAME', 'ag_jobs', 'total_production_USD',
       'total_production_t', 'other_share_production_USD',
       'share_vlarge_field', 'lon', 'lat', 'ISO3',
       'country_other_share_production_USD', 'country_share_vlarge_field',
       'country_capital_intensity_USD_per_million_USD',
       'country_capital_intensity_USD_per_million_tonne']

capital_merge_country = capital_merge_country.drop(columns=col_to_drop_cap)
labor_merge_country = labor_merge_country.drop(columns=col_to_drop_lab)

In [81]:
### Save
capital_merge_country.to_csv(save_path_capital_rtv, index=False)
labor_merge_country.to_csv(save_path_labor_rtv, index=False)